In [7]:
import pandas as pd
import numpy as np
import gc

def optimize_memory(df, date_cols=None, category_threshold=0.05):
    """
    Optimizes DataFrame memory usage.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    date_cols : list or None
        List of columns to convert to datetime.
        If None, columns containing 'date' are automatically detected.
    category_threshold : float
        Convert object columns to category if:
        unique_values / total_rows < threshold

    Returns
    -------
    pandas.DataFrame
    """

    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory before optimization : {start_mem:.2f} MB")

    # ------------------------------
    # Convert date columns
    # ------------------------------
    if date_cols is None:
        date_cols = [c for c in df.columns if "date" in c.lower()]

    for col in date_cols:
        if col in df.columns:
            try:
                df[col] = pd.to_datetime(df[col], errors="coerce")
                print(f"✓ {col} -> datetime64")
            except:
                pass

    # ------------------------------
    # Process every column
    # ------------------------------
    for col in df.columns:

        # Skip datetime columns
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        # Integer columns
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")

        # Float columns
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="float")

        # Object columns
        elif pd.api.types.is_object_dtype(df[col]):

            unique_ratio = df[col].nunique(dropna=False) / len(df)

            if unique_ratio < category_threshold:
                df[col] = df[col].astype("category")
                print(f"✓ {col} -> category")

    gc.collect()

    end_mem = df.memory_usage(deep=True).sum() / 1024**2

    print("\n-----------------------------------------")
    print(f"Memory after optimization  : {end_mem:.2f} MB")
    print(f"Memory reduced             : {start_mem-end_mem:.2f} MB")
    print(f"Reduction                  : {100*(start_mem-end_mem)/start_mem:.2f}%")
    print("-----------------------------------------")

    return df

In [8]:
sales_subset = pd.read_csv("/kaggle/input/datasets/roshanshinde803/m5-reduced/sales_subset_15000 (1).csv")

In [9]:
sales_subset = optimize_memory(sales_subset)

Memory before optimization : 224.03 MB
✓ dept_id -> category
✓ cat_id -> category
✓ store_id -> category
✓ state_id -> category

-----------------------------------------
Memory after optimization  : 38.65 MB
Memory reduced             : 185.39 MB
Reduction                  : 82.75%
-----------------------------------------


In [10]:
sales_subset.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,FOODS_3_180_CA_1_validation,FOODS_3_180,FOODS_3,FOODS,CA_1,CA,0,0,0,0,...,0,0,1,2,0,0,0,0,1,0
1,HOUSEHOLD_2_383_CA_3_validation,HOUSEHOLD_2_383,HOUSEHOLD_2,HOUSEHOLD,CA_3,CA,2,0,2,0,...,0,2,0,1,0,0,0,0,0,1
2,FOODS_3_409_CA_3_validation,FOODS_3_409,FOODS_3,FOODS,CA_3,CA,0,0,0,0,...,0,0,1,0,0,1,1,2,0,0
3,FOODS_1_097_CA_2_validation,FOODS_1_097,FOODS_1,FOODS,CA_2,CA,0,0,0,0,...,0,1,1,2,2,0,2,2,1,0
4,HOBBIES_1_272_TX_2_validation,HOBBIES_1_272,HOBBIES_1,HOBBIES,TX_2,TX,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
calendar = pd.read_csv("/kaggle/input/datasets/roshanshinde803/m5dataset/calendar.csv")

In [12]:
calendar = optimize_memory(calendar)

Memory before optimization : 0.67 MB
✓ date -> datetime64
✓ weekday -> category
✓ event_name_1 -> category
✓ event_type_1 -> category
✓ event_name_2 -> category
✓ event_type_2 -> category

-----------------------------------------
Memory after optimization  : 0.15 MB
Memory reduced             : 0.52 MB
Reduction                  : 77.99%
-----------------------------------------


In [13]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [14]:
sales_subset.drop(columns=[ 'cat_id', 'state_id'], inplace=True)

In [15]:
sales_subset.head()

,id,item_id,dept_id,store_id,d_1,d_2,d_3,d_4,d_5,d_6,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,FOODS_3_180_CA_1_validation,FOODS_3_180,FOODS_3,CA_1,0,0,0,0,0,0,...,0,0,1,2,0,0,0,0,1,0
1,HOUSEHOLD_2_383_CA_3_validation,HOUSEHOLD_2_383,HOUSEHOLD_2,CA_3,2,0,2,0,2,0,...,0,2,0,1,0,0,0,0,0,1
2,FOODS_3_409_CA_3_validation,FOODS_3_409,FOODS_3,CA_3,0,0,0,0,0,0,...,0,0,1,0,0,1,1,2,0,0
3,FOODS_1_097_CA_2_validation,FOODS_1_097,FOODS_1,CA_2,0,0,0,0,0,0,...,0,1,1,2,2,0,2,2,1,0
4,HOBBIES_1_272_TX_2_validation,HOBBIES_1_272,HOBBIES_1,TX_2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
d_cols = [c for c in sales_subset.columns if c.startswith("d_")]

sales_long = sales_subset.melt(
    id_vars=[
        "item_id",
        "dept_id",
        "store_id"
    ],
    value_vars=d_cols,
    var_name="d",
    value_name="sales"
)

In [17]:
sales_long.head()

,item_id,dept_id,store_id,d,sales
0,FOODS_3_180,FOODS_3,CA_1,d_1,0
1,HOUSEHOLD_2_383,HOUSEHOLD_2,CA_3,d_1,2
2,FOODS_3_409,FOODS_3,CA_3,d_1,0
3,FOODS_1_097,FOODS_1,CA_2,d_1,0
4,HOBBIES_1_272,HOBBIES_1,TX_2,d_1,0


In [18]:
sales_long.shape


(28695000, 5)

In [19]:
sales_calendar = sales_long.merge(
    calendar,
    on='d',
    how='left'
)

In [20]:
sales_calendar.head(30)

,item_id,dept_id,store_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,FOODS_3_180,FOODS_3,CA_1,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,HOUSEHOLD_2_383,HOUSEHOLD_2,CA_3,d_1,2,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,FOODS_3_409,FOODS_3,CA_3,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,FOODS_1_097,FOODS_1,CA_2,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
4,HOBBIES_1_272,HOBBIES_1,TX_2,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
5,HOUSEHOLD_1_064,HOUSEHOLD_1,CA_4,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
6,FOODS_2_368,FOODS_2,TX_2,d_1,14,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
7,HOUSEHOLD_1_395,HOUSEHOLD_1,TX_3,d_1,5,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
8,HOUSEHOLD_1_537,HOUSEHOLD_1,CA_1,d_1,3,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
9,HOUSEHOLD_2_410,HOUSEHOLD_2,TX_2,d_1,1,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0


In [21]:
sell_prices = pd.read_csv("/kaggle/input/datasets/roshanshinde803/m5dataset/sell_prices.csv")

In [22]:
sell_prices = optimize_memory(sell_prices)

Memory before optimization : 853.13 MB
✓ store_id -> category
✓ item_id -> category

-----------------------------------------
Memory after optimization  : 58.96 MB
Memory reduced             : 794.17 MB
Reduction                  : 93.09%
-----------------------------------------


In [23]:
sell_prices

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26
...,...,...,...,...
6841116,WI_3,FOODS_3_827,11617,1.00
6841117,WI_3,FOODS_3_827,11618,1.00
6841118,WI_3,FOODS_3_827,11619,1.00
6841119,WI_3,FOODS_3_827,11620,1.00


In [24]:
def merge_sell_prices(sales_calendar, sell_prices):
    """
    Merge sell_prices.csv with sales_calendar dataframe.

    Parameters
    ----------
    sales_calendar : pd.DataFrame
        Data after merging sales with calendar.
    sell_prices : pd.DataFrame
        Original sell_prices.csv

    Returns
    -------
    pd.DataFrame
        Merged dataframe.
    """

    print("=" * 50)
    print("Merging sell_prices...")
    print("=" * 50)

    before_shape = sales_calendar.shape

    merged = sales_calendar.merge(
        sell_prices,
        how="left",
        on=["store_id", "item_id", "wm_yr_wk"]
    )

    print(f"Before Merge : {before_shape}")
    print(f"After Merge  : {merged.shape}")

    missing = merged["sell_price"].isna().sum()

    print(f"Missing sell_price values : {missing:,}")

    if missing == 0:
        print("✅ Merge Successful!")
    else:
        print(f"⚠️ Warning: {missing:,} rows have missing prices.")

    return merged

In [25]:
print(sales_calendar[['d','wm_yr_wk']].head(15))

      d  wm_yr_wk
0   d_1     11101
1   d_1     11101
2   d_1     11101
3   d_1     11101
4   d_1     11101
5   d_1     11101
6   d_1     11101
7   d_1     11101
8   d_1     11101
9   d_1     11101
10  d_1     11101
11  d_1     11101
12  d_1     11101
13  d_1     11101
14  d_1     11101


In [26]:
sales_final = merge_sell_prices(
    sales_calendar,
    sell_prices
)

Merging sell_prices...
Before Merge : (28695000, 18)
After Merge  : (28695000, 19)
Missing sell_price values : 6,075,650
⚠️ Warning: 6,075,650 rows have missing prices.


In [27]:
sales_final = optimize_memory(sales_final)

Memory before optimization : 3999.99 MB
✓ date -> datetime64
✓ item_id -> category
✓ d -> category

-----------------------------------------
Memory after optimization  : 930.84 MB
Memory reduced             : 3069.14 MB
Reduction                  : 76.73%
-----------------------------------------


### fill using the previous known price for the same product in the same store:

In [28]:
sales_final["sell_price"] = (
    sales_final
    .groupby(["item_id", "store_id"])["sell_price"]
    .transform(lambda x: x.ffill().bfill())
)

/tmp/ipykernel_58/854428906.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["item_id", "store_id"])["sell_price"]


In [29]:
sales_final

,item_id,dept_id,store_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_3_180,FOODS_3,CA_1,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.78
1,HOUSEHOLD_2_383,HOUSEHOLD_2,CA_3,d_1,2,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,3.97
2,FOODS_3_409,FOODS_3,CA_3,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.78
3,FOODS_1_097,FOODS_1,CA_2,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.98
4,HOBBIES_1_272,HOBBIES_1,TX_2,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,8.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28694995,FOODS_2_373,FOODS_2,CA_2,d_1913,3,2016-04-24,11613,Sunday,2,4,2016,NaN,NaN,NaN,NaN,0,0,0,5.98
28694996,FOODS_2_117,FOODS_2,WI_3,d_1913,2,2016-04-24,11613,Sunday,2,4,2016,NaN,NaN,NaN,NaN,0,0,0,5.97
28694997,FOODS_3_688,FOODS_3,TX_1,d_1913,13,2016-04-24,11613,Sunday,2,4,2016,NaN,NaN,NaN,NaN,0,0,0,1.88
28694998,HOUSEHOLD_1_099,HOUSEHOLD_1,WI_3,d_1913,1,2016-04-24,11613,Sunday,2,4,2016,NaN,NaN,NaN,NaN,0,0,0,2.37


In [30]:
sales_final["sell_price"].isna().sum()

np.int64(0)

In [31]:
sales_final = sales_final.drop(
    columns=["d", "weekday", "month", "year"],
    errors="ignore"
)

In [32]:
sales_final

,item_id,dept_id,store_id,sales,date,wm_yr_wk,wday,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_3_180,FOODS_3,CA_1,0,2011-01-29,11101,1,NaN,NaN,NaN,NaN,0,0,0,2.78
1,HOUSEHOLD_2_383,HOUSEHOLD_2,CA_3,2,2011-01-29,11101,1,NaN,NaN,NaN,NaN,0,0,0,3.97
2,FOODS_3_409,FOODS_3,CA_3,0,2011-01-29,11101,1,NaN,NaN,NaN,NaN,0,0,0,2.78
3,FOODS_1_097,FOODS_1,CA_2,0,2011-01-29,11101,1,NaN,NaN,NaN,NaN,0,0,0,0.98
4,HOBBIES_1_272,HOBBIES_1,TX_2,0,2011-01-29,11101,1,NaN,NaN,NaN,NaN,0,0,0,8.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28694995,FOODS_2_373,FOODS_2,CA_2,3,2016-04-24,11613,2,NaN,NaN,NaN,NaN,0,0,0,5.98
28694996,FOODS_2_117,FOODS_2,WI_3,2,2016-04-24,11613,2,NaN,NaN,NaN,NaN,0,0,0,5.97
28694997,FOODS_3_688,FOODS_3,TX_1,13,2016-04-24,11613,2,NaN,NaN,NaN,NaN,0,0,0,1.88
28694998,HOUSEHOLD_1_099,HOUSEHOLD_1,WI_3,1,2016-04-24,11613,2,NaN,NaN,NaN,NaN,0,0,0,2.37


In [33]:
sales_final["event_name_2"].unique()

[NaN, 'Easter', 'Cinco De Mayo', 'OrthodoxEaster', 'Father's day']
Categories (4, object): ['Cinco De Mayo', 'Easter', 'Father's day', 'OrthodoxEaster']

In [34]:
sales_final = sales_final.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

In [35]:
sales_final

,item_id,dept_id,store_id,sales,date,wm_yr_wk,wday,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001,FOODS_1,CA_1,3,2011-01-29,11101,1,NaN,NaN,NaN,NaN,0,0,0,2.00
1,FOODS_1_001,FOODS_1,CA_1,0,2011-01-30,11101,2,NaN,NaN,NaN,NaN,0,0,0,2.00
2,FOODS_1_001,FOODS_1,CA_1,0,2011-01-31,11101,3,NaN,NaN,NaN,NaN,0,0,0,2.00
3,FOODS_1_001,FOODS_1,CA_1,1,2011-02-01,11101,4,NaN,NaN,NaN,NaN,1,1,0,2.00
4,FOODS_1_001,FOODS_1,CA_1,4,2011-02-02,11101,5,NaN,NaN,NaN,NaN,1,0,1,2.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28694995,HOUSEHOLD_2_516,HOUSEHOLD_2,WI_3,0,2016-04-20,11612,5,NaN,NaN,NaN,NaN,0,0,0,5.94
28694996,HOUSEHOLD_2_516,HOUSEHOLD_2,WI_3,0,2016-04-21,11612,6,NaN,NaN,NaN,NaN,0,0,0,5.94
28694997,HOUSEHOLD_2_516,HOUSEHOLD_2,WI_3,0,2016-04-22,11612,7,NaN,NaN,NaN,NaN,0,0,0,5.94
28694998,HOUSEHOLD_2_516,HOUSEHOLD_2,WI_3,0,2016-04-23,11613,1,NaN,NaN,NaN,NaN,0,0,0,5.94


**Create one binary feature: for snap as IF STORE IS OF CA THEN SNAP IN CA MATTERS NO OTHER SNAPS MATTER**

In [36]:
import numpy as np

sales_final["snap"] = np.select(
    [
        sales_final["store_id"].str.startswith("CA"),
        sales_final["store_id"].str.startswith("TX"),
        sales_final["store_id"].str.startswith("WI"),
    ],
    [
        sales_final["snap_CA"],
        sales_final["snap_TX"],
        sales_final["snap_WI"],
    ],
    default=0,
)

In [37]:
sales_final.drop(
    columns=["snap_CA", "snap_TX", "snap_WI"],
    inplace=True
)

In [38]:
sales_final

,item_id,dept_id,store_id,sales,date,wm_yr_wk,wday,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,snap
0,FOODS_1_001,FOODS_1,CA_1,3,2011-01-29,11101,1,NaN,NaN,NaN,NaN,2.00,0
1,FOODS_1_001,FOODS_1,CA_1,0,2011-01-30,11101,2,NaN,NaN,NaN,NaN,2.00,0
2,FOODS_1_001,FOODS_1,CA_1,0,2011-01-31,11101,3,NaN,NaN,NaN,NaN,2.00,0
3,FOODS_1_001,FOODS_1,CA_1,1,2011-02-01,11101,4,NaN,NaN,NaN,NaN,2.00,1
4,FOODS_1_001,FOODS_1,CA_1,4,2011-02-02,11101,5,NaN,NaN,NaN,NaN,2.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
28694995,HOUSEHOLD_2_516,HOUSEHOLD_2,WI_3,0,2016-04-20,11612,5,NaN,NaN,NaN,NaN,5.94,0
28694996,HOUSEHOLD_2_516,HOUSEHOLD_2,WI_3,0,2016-04-21,11612,6,NaN,NaN,NaN,NaN,5.94,0
28694997,HOUSEHOLD_2_516,HOUSEHOLD_2,WI_3,0,2016-04-22,11612,7,NaN,NaN,NaN,NaN,5.94,0
28694998,HOUSEHOLD_2_516,HOUSEHOLD_2,WI_3,0,2016-04-23,11613,1,NaN,NaN,NaN,NaN,5.94,0


In [39]:
for lag in [1,7,14,28]:
    sales_final[f"lag_{lag}"] = (
        sales_final
        .groupby(["item_id","store_id"])["sales"]
        .shift(lag)
    )

/tmp/ipykernel_58/3178641741.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["item_id","store_id"])["sales"]
/tmp/ipykernel_58/3178641741.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["item_id","store_id"])["sales"]
/tmp/ipykernel_58/3178641741.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["item_id","store_id"])["sales"]
/tmp/ipykernel_58/3178641741.py:4: FutureWarning: The d

In [40]:
sales_final.head(20)

,item_id,dept_id,store_id,sales,date,wm_yr_wk,wday,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,snap,lag_1,lag_7,lag_14,lag_28
0,FOODS_1_001,FOODS_1,CA_1,3,2011-01-29,11101,1,NaN,NaN,NaN,NaN,2.0,0,NaN,NaN,NaN,NaN
1,FOODS_1_001,FOODS_1,CA_1,0,2011-01-30,11101,2,NaN,NaN,NaN,NaN,2.0,0,3.0,NaN,NaN,NaN
2,FOODS_1_001,FOODS_1,CA_1,0,2011-01-31,11101,3,NaN,NaN,NaN,NaN,2.0,0,0.0,NaN,NaN,NaN
3,FOODS_1_001,FOODS_1,CA_1,1,2011-02-01,11101,4,NaN,NaN,NaN,NaN,2.0,1,0.0,NaN,NaN,NaN
4,FOODS_1_001,FOODS_1,CA_1,4,2011-02-02,11101,5,NaN,NaN,NaN,NaN,2.0,1,1.0,NaN,NaN,NaN
5,FOODS_1_001,FOODS_1,CA_1,2,2011-02-03,11101,6,NaN,NaN,NaN,NaN,2.0,1,4.0,NaN,NaN,NaN
6,FOODS_1_001,FOODS_1,CA_1,0,2011-02-04,11101,7,NaN,NaN,NaN,NaN,2.0,1,2.0,NaN,NaN,NaN
7,FOODS_1_001,FOODS_1,CA_1,2,2011-02-05,11102,1,NaN,NaN,NaN,NaN,2.0,1,0.0,3.0,NaN,NaN
8,FOODS_1_001,FOODS_1,CA_1,0,2011-02-06,11102,2,SuperBowl,Sporting,NaN,NaN,2.0,1,2.0,0.0,NaN,NaN
9,FOODS_1_001,FOODS_1,CA_1,0,2011-02-07,11102,3,NaN,NaN,NaN,NaN,2.0,1,0.0,0.0,NaN,NaN


In [41]:
sales_final["rolling_mean_7"] = (
    sales_final
    .groupby(["item_id","store_id"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

sales_final["rolling_std_7"] = (
    sales_final
    .groupby(["item_id","store_id"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

/tmp/ipykernel_58/3223348424.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["item_id","store_id"])["sales"]
/tmp/ipykernel_58/3223348424.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["item_id","store_id"])["sales"]


In [42]:
sales_final = optimize_memory(sales_final)

Memory before optimization : 2025.31 MB
✓ date -> datetime64

-----------------------------------------
Memory after optimization  : 1368.53 MB
Memory reduced             : 656.78 MB
Reduction                  : 32.43%
-----------------------------------------


**Price Difference**

In [43]:
sales_final["price_diff"] = (
    sales_final
    .groupby(["item_id","store_id"])["sell_price"]
    .diff()
)

/tmp/ipykernel_58/2895167928.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["item_id","store_id"])["sell_price"]


### PRICE MOMENTUM

In [44]:
sales_final["price_momentum"] = (
    sales_final["sell_price"] /
    sales_final.groupby(["item_id","store_id"])["sell_price"].shift(1)
)

/tmp/ipykernel_58/1680249870.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sales_final.groupby(["item_id","store_id"])["sell_price"].shift(1)



### Trend captures the long-term movement of sales.





In [45]:
sales_final["trend"] = (
    sales_final
    .groupby(["item_id","store_id"])
    .cumcount()
)

/tmp/ipykernel_58/3372883182.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["item_id","store_id"])


## Seasonality

In [46]:
sales_final["week"] = sales_final["date"].dt.isocalendar().week.astype(int)

sales_final["quarter"] = sales_final["date"].dt.quarter

In [47]:
sales_final[sales_final["date"].isna()][["wm_yr_wk"]].head()

,wm_yr_wk


In [48]:
print(sales_long["d"].head())
print(sales_long["d"].tail())

0    d_1
1    d_1
2    d_1
3    d_1
4    d_1
Name: d, dtype: object
28694995    d_1913
28694996    d_1913
28694997    d_1913
28694998    d_1913
28694999    d_1913
Name: d, dtype: object


In [49]:
print(calendar["d"].head())
print(calendar["d"].tail())

0    d_1
1    d_2
2    d_3
3    d_4
4    d_5
Name: d, dtype: object
1964    d_1965
1965    d_1966
1966    d_1967
1967    d_1968
1968    d_1969
Name: d, dtype: object


In [50]:
sales_final.describe()

,sales,date,wm_yr_wk,wday,sell_price,snap,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,price_diff,price_momentum,trend,week,quarter
count,2.869500e+07,28695000,2.869500e+07,2.869500e+07,2.869500e+07,2.869500e+07,2.868000e+07,2.859000e+07,2.848500e+07,2.827500e+07,2.859000e+07,2.859000e+07,2.868000e+07,2.868000e+07,2.869500e+07,2.869500e+07,2.869500e+07
mean,1.120762e+00,2013-09-11 00:00:00,1.133919e+04,3.997386e+00,4.422531e+00,3.293257e-01,1.120488e+00,1.119957e+00,1.118967e+00,1.116768e+00,1.120502e+00,8.995261e-01,4.904010e-05,1.000117e+00,9.560000e+02,2.587088e+01,2.452692e+00
min,0.000000e+00,2011-01-29 00:00:00,1.110100e+04,1.000000e+00,1.000000e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-2.722000e+01,3.355705e-03,0.000000e+00,1.000000e+00,1.000000e+00
25%,0.000000e+00,2012-05-21 00:00:00,1.121700e+04,2.000000e+00,2.180000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,4.780000e+02,1.300000e+01,1.000000e+00
50%,0.000000e+00,2013-09-11 00:00:00,1.133300e+04,4.000000e+00,3.370000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.857143e-01,4.879500e-01,0.000000e+00,1.000000e+00,9.560000e+02,2.500000e+01,2.000000e+00
75%,1.000000e+00,2015-01-02 00:00:00,1.144800e+04,6.000000e+00,5.780000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.133893e+00,0.000000e+00,1.000000e+00,1.434000e+03,3.900000e+01,3.000000e+00
max,6.340000e+02,2016-04-24 00:00:00,1.161300e+04,7.000000e+00,3.418000e+01,1.000000e+00,6.340000e+02,6.340000e+02,6.340000e+02,6.340000e+02,2.984286e+02,2.383281e+02,2.722000e+01,2.980000e+02,1.912000e+03,5.300000e+01,4.000000e+00
std,3.789268e+00,NaN,1.503742e+02,2.000652e+00,3.400668e+00,4.699684e-01,3.661304e+00,3.662683e+00,3.663032e+00,3.664245e+00,3.194983e+00,1.733118e+00,3.971598e-02,3.853334e-01,5.522355e+02,1.512504e+01,1.125192e+00


### Drop the rows with missing lag/rolling features:


In [51]:
sales_final = sales_final.dropna(
    subset=[
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_std_7"
    ]
).reset_index(drop=True)

In [52]:
sales_final = optimize_memory(sales_final)

Memory before optimization : 2103.53 MB
✓ date -> datetime64

-----------------------------------------
Memory after optimization  : 1672.09 MB
Memory reduced             : 431.44 MB
Reduction                  : 20.51%
-----------------------------------------


In [53]:
sales_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28275000 entries, 0 to 28274999
Data columns (total 24 columns):
 #   Column          Dtype         
---  ------          -----         
 0   item_id         category      
 1   dept_id         category      
 2   store_id        category      
 3   sales           int16         
 4   date            datetime64[ns]
 5   wm_yr_wk        int16         
 6   wday            int8          
 7   event_name_1    category      
 8   event_type_1    category      
 9   event_name_2    category      
 10  event_type_2    category      
 11  sell_price      float32       
 12  snap            int8          
 13  lag_1           float32       
 14  lag_7           float32       
 15  lag_14          float32       
 16  lag_28          float32       
 17  rolling_mean_7  float32       
 18  rolling_std_7   float32       
 19  price_diff      float32       
 20  price_momentum  float32       
 21  trend           int16         
 22  week            

In [54]:
sales_final.head()

,item_id,dept_id,store_id,sales,date,wm_yr_wk,wday,event_name_1,event_type_1,event_name_2,...,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,price_diff,price_momentum,trend,week,quarter
0,FOODS_1_001,FOODS_1,CA_1,2,2011-02-26,11105,1,NaN,NaN,NaN,...,1.0,3.0,3.0,1.857143,1.214986,0.0,1.0,28,8,1
1,FOODS_1_001,FOODS_1,CA_1,2,2011-02-27,11105,2,NaN,NaN,NaN,...,2.0,0.0,0.0,2.000000,1.154701,0.0,1.0,29,8,1
2,FOODS_1_001,FOODS_1,CA_1,0,2011-02-28,11105,3,NaN,NaN,NaN,...,0.0,2.0,0.0,2.000000,1.154701,0.0,1.0,30,9,1
3,FOODS_1_001,FOODS_1,CA_1,2,2011-03-01,11105,4,NaN,NaN,NaN,...,2.0,1.0,1.0,2.000000,1.154701,0.0,1.0,31,9,1
4,FOODS_1_001,FOODS_1,CA_1,1,2011-03-02,11105,5,NaN,NaN,NaN,...,2.0,2.0,4.0,2.000000,1.154701,0.0,1.0,32,9,1


### Fill missing event values:

In [55]:
event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in event_cols:
    sales_final[col] = sales_final[col].cat.add_categories(["None"])
    sales_final[col] = sales_final[col].fillna("None")

In [56]:
sales_final.head()

,item_id,dept_id,store_id,sales,date,wm_yr_wk,wday,event_name_1,event_type_1,event_name_2,...,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,price_diff,price_momentum,trend,week,quarter
0,FOODS_1_001,FOODS_1,CA_1,2,2011-02-26,11105,1,None,None,None,...,1.0,3.0,3.0,1.857143,1.214986,0.0,1.0,28,8,1
1,FOODS_1_001,FOODS_1,CA_1,2,2011-02-27,11105,2,None,None,None,...,2.0,0.0,0.0,2.000000,1.154701,0.0,1.0,29,8,1
2,FOODS_1_001,FOODS_1,CA_1,0,2011-02-28,11105,3,None,None,None,...,0.0,2.0,0.0,2.000000,1.154701,0.0,1.0,30,9,1
3,FOODS_1_001,FOODS_1,CA_1,2,2011-03-01,11105,4,None,None,None,...,2.0,1.0,1.0,2.000000,1.154701,0.0,1.0,31,9,1
4,FOODS_1_001,FOODS_1,CA_1,1,2011-03-02,11105,5,None,None,None,...,2.0,2.0,4.0,2.000000,1.154701,0.0,1.0,32,9,1


### spliting first for training and testing so that encoding can be applied


In [57]:
sales_final = sales_final.sort_values("date").reset_index(drop=True)

# Get all unique dates in chronological order
unique_dates = sorted(sales_final["date"].unique())

# Split indices
n_dates = len(unique_dates)

train_end = int(n_dates * 0.70)
val_end = int(n_dates * 0.85)

# Date ranges
train_dates = unique_dates[:train_end]
val_dates = unique_dates[train_end:val_end]
test_dates = unique_dates[val_end:]

# Create datasets
train_df = sales_final[sales_final["date"].isin(train_dates)].copy()
val_df = sales_final[sales_final["date"].isin(val_dates)].copy()
test_df = sales_final[sales_final["date"].isin(test_dates)].copy()

# Check sizes
print(f"Train Shape: {train_df.shape}")
print(f"Validation Shape: {val_df.shape}")
print(f"Test Shape: {test_df.shape}")

print(f"Train Dates: {train_df['date'].min()} --> {train_df['date'].max()}")
print(f"Validation Dates: {val_df['date'].min()} --> {val_df['date'].max()}")
print(f"Test Dates: {test_df['date'].min()} --> {test_df['date'].max()}")

Train Shape: (19785000, 24)
Validation Shape: (4245000, 24)
Test Shape: (4245000, 24)
Train Dates: 2011-02-26 00:00:00 --> 2014-10-06 00:00:00
Validation Dates: 2014-10-07 00:00:00 --> 2015-07-16 00:00:00
Test Dates: 2015-07-17 00:00:00 --> 2016-04-24 00:00:00


### Separate Features and Target

In [58]:
TARGET = "sales"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]



# Label encoding and ohe 
### using label as there is no order but still the item_id and event_type_1 can't be ohe as it has 3049 products and 31 unique events (did ordinal as similar to label encoding)

In [59]:
from sklearn.preprocessing import OrdinalEncoder

# Categorical columns
cat_cols = [
    "item_id",
    "dept_id",
    "store_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

# Create encoder
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# Fit only on training data
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols].astype(str))

# Transform validation and test data
X_val[cat_cols] = encoder.transform(X_val[cat_cols].astype(str))
X_test[cat_cols] = encoder.transform(X_test[cat_cols].astype(str))

# Downcast to reduce memory
X_train[cat_cols] = X_train[cat_cols].astype("int16")
X_val[cat_cols] = X_val[cat_cols].astype("int16")
X_test[cat_cols] = X_test[cat_cols].astype("int16")

print("✅ Label Encoding Completed!")

print("Train Shape :", X_train.shape)
print("Validation Shape :", X_val.shape)
print("Test Shape :", X_test.shape)

✅ Label Encoding Completed!
Train Shape : (19785000, 23)
Validation Shape : (4245000, 23)
Test Shape : (4245000, 23)


In [60]:
X_train.head()

,item_id,dept_id,store_id,date,wm_yr_wk,wday,event_name_1,event_type_1,event_name_2,event_type_2,...,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,price_diff,price_momentum,trend,week,quarter
0,0,0,0,2011-02-26,11105,1,19,2,3,1,...,1.0,3.0,3.0,1.857143,1.214986,0.0,1.0,28,8,1
1,2007,5,8,2011-02-26,11105,1,19,2,3,1,...,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,28,8,1
2,2007,5,9,2011-02-26,11105,1,19,2,3,1,...,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,28,8,1
3,2008,5,1,2011-02-26,11105,1,19,2,3,1,...,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,28,8,1
4,2008,5,2,2011-02-26,11105,1,19,2,3,1,...,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,28,8,1


In [61]:
X_train.drop(columns=["date"], inplace=True)
X_val.drop(columns=["date"], inplace=True)
X_test.drop(columns=["date"], inplace=True)

In [62]:
print(y_train.describe())

count    1.978500e+07
mean     1.071639e+00
std      3.875817e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      6.340000e+02
Name: sales, dtype: float64


# Train a baseline XGBoost

In [63]:
from xgboost import XGBRegressor

In [64]:
import xgboost
print(xgboost.__version__)
from xgboost import XGBRegressor

3.2.0


In [65]:
model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="rmse",
    device = "cuda",
    early_stopping_rounds=50,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)


[0]	validation_0-rmse:3.37120
[50]	validation_0-rmse:2.03866
[100]	validation_0-rmse:2.02110
[150]	validation_0-rmse:2.02133
[177]	validation_0-rmse:2.02191


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=50,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_parallel_tree=None, ...)

In [66]:
model.save_model("xgboost_model.json")

In [67]:
print(model.best_iteration)
print(model.best_score)

127
2.0194406087983667


In [68]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("Test RMSE:", rmse)
print("Test MAE :", mae)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [17:42:03] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Test RMSE: 2.0482004547887462
Test MAE : 0.9543305039405823


In [69]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("R²:", r2)

R²: 0.6879847049713135


In [71]:
from scipy.stats import randint, uniform

param_dist = {
    "max_depth": randint(4, 12),
    "learning_rate": uniform(0.01, 0.09),      # 0.01 - 0.10
    "min_child_weight": randint(1, 10),
    "subsample": uniform(0.6, 0.4),            # 0.6 - 1.0
    "colsample_bytree": uniform(0.6, 0.4),     # 0.6 - 1.0
    "gamma": uniform(0, 5),
    "reg_alpha": uniform(0, 1),
    "reg_lambda": uniform(0.5, 2)
}

In [74]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    device="cuda",
    eval_metric="rmse",
    n_estimators=300,
    random_state=42
)

In [76]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1
)

In [77]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.749816047538945, gamma=4.75357153204958, learning_rate=0.07587945476302645, max_depth=8, min_child_weight=5, reg_alpha=0.15601864044243652, reg_lambda=0.8119890406724053, subsample=0.6232334448672797; total time=  34.4s
[CV] END colsample_bytree=0.749816047538945, gamma=4.75357153204958, learning_rate=0.07587945476302645, max_depth=8, min_child_weight=5, reg_alpha=0.15601864044243652, reg_lambda=0.8119890406724053, subsample=0.6232334448672797; total time=  34.9s
[CV] END colsample_bytree=0.749816047538945, gamma=4.75357153204958, learning_rate=0.07587945476302645, max_depth=8, min_child_weight=5, reg_alpha=0.15601864044243652, reg_lambda=0.8119890406724053, subsample=0.6232334448672797; total time=  33.3s
[CV] END colsample_bytree=0.9464704583099741, gamma=3.005575058716044, learning_rate=0.07372653200164409, max_depth=9, min_child_weight=5, reg_alpha=0.9699098521619943, reg_lambda=2.1648852816008

KeyboardInterrupt: 

### HYPERTUNNING PARAMETERS

In [80]:
X_tune = X_train.sample(
    n=2_000_000,
    random_state=42
)

y_tune = y_train.loc[X_tune.index]

In [81]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    device="cuda",
    eval_metric="rmse",
    n_estimators=300,
    random_state=42
)

In [82]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=5,          # Reduced from 20
    cv=2,              # Reduced from 3
    scoring="neg_root_mean_squared_error",
    verbose=2,
    random_state=42,
    n_jobs=1
)

In [83]:
random_search.fit(X_tune, y_tune)

Fitting 2 folds for each of 5 candidates, totalling 10 fits
[CV] END colsample_bytree=0.749816047538945, gamma=4.75357153204958, learning_rate=0.07587945476302645, max_depth=8, min_child_weight=5, reg_alpha=0.15601864044243652, reg_lambda=0.8119890406724053, subsample=0.6232334448672797; total time=   3.7s
[CV] END colsample_bytree=0.749816047538945, gamma=4.75357153204958, learning_rate=0.07587945476302645, max_depth=8, min_child_weight=5, reg_alpha=0.15601864044243652, reg_lambda=0.8119890406724053, subsample=0.6232334448672797; total time=   3.6s
[CV] END colsample_bytree=0.9464704583099741, gamma=3.005575058716044, learning_rate=0.07372653200164409, max_depth=9, min_child_weight=5, reg_alpha=0.9699098521619943, reg_lambda=2.1648852816008435, subsample=0.6849356442713105; total time=   3.9s
[CV] END colsample_bytree=0.9464704583099741, gamma=3.005575058716044, learning_rate=0.07372653200164409, max_depth=9, min_child_weight=5, reg_alpha=0.9699098521619943, reg_lambda=2.1648852816008

RandomizedSearchCV(cv=2,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device='cuda',
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric='rmse',
                                          feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constr...
                                        'min_child_weight': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x77fb94be3920>,
                                        'reg_alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x77fb94be2ab0>,
                                        'reg_lambda': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x77fb94be25a0>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x77fb94be3ce0>},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=2)

In [84]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest RMSE:")
print(-random_search.best_score_)

Best Parameters:
{'colsample_bytree': np.float64(0.6727299868828402), 'gamma': np.float64(0.9170225492671691), 'learning_rate': np.float64(0.037381801866358394), 'max_depth': 9, 'min_child_weight': 9, 'reg_alpha': np.float64(0.2912291401980419), 'reg_lambda': np.float64(1.723705789444759), 'subsample': np.float64(0.6557975442608167)}

Best RMSE:
2.073933720588684


In [85]:
best_params = random_search.best_params_

final_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    device="cuda",
    eval_metric="rmse",
    random_state=42,
    **best_params
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

[0]	validation_0-rmse:3.40077
[50]	validation_0-rmse:2.07055
[99]	validation_0-rmse:2.01521


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.6727299868828402), device='cuda',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=np.float64(0.9170225492671691), grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=np.float64(0.037381801866358394), max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=9, max_leaves=None,
             min_child_weight=9, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, ...)

In [86]:
final_model.save_model("final_xgboost_model.json")

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [87]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
    max_error
)
import numpy as np

# Predictions
y_pred = final_model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
evs = explained_variance_score(y_test, y_pred)
max_err = max_error(y_test, y_pred)

# MAPE (avoid division by zero)
mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

# Print Results
print("=" * 45)
print("         XGBOOST MODEL EVALUATION")
print("=" * 45)

print(f"MAE                 : {mae:.6f}")
print(f"MSE                 : {mse:.6f}")
print(f"RMSE                : {rmse:.6f}")
print(f"R² Score            : {r2:.6f}")
print(f"Explained Variance  : {evs:.6f}")
print(f"MAPE (%)            : {mape:.2f}%")
print(f"Maximum Error       : {max_err:.6f}")

print("=" * 45)

         XGBOOST MODEL EVALUATION
MAE                 : 0.962660
MSE                 : 4.190317
RMSE                : 2.047026
R² Score            : 0.688342
Explained Variance  : 0.688350
MAPE (%)            : 55.85%
Maximum Error       : 202.794525
